# End-to-End MLP Workflow

This notebook is a fully worked example of how to use `portfolio_toolkit` with a basic MLP-style forecasting model.

It covers the full notebook-first workflow:

1. load a shared dataset
2. add built-in toolkit features
3. engineer new custom notebook-local features
4. build forward return / alpha / volatility targets
5. split into train / validation / test using the shared split rules
6. define and train a small PyTorch MLP regressor
7. emit a standardized prediction table
8. turn predictions into a `PortfolioWeights` object
9. run the shared backtest
10. write reports and artifacts
11. log everything to MLflow

This is intentionally simple and heavily commented. The idea is that a teammate can copy this notebook, replace the model body, keep the shared data/evaluation layer, and still be comparable to everyone else.

## Running This Notebook In Colab

If you want to run this notebook in Google Colab, start by cloning the repo into the Colab session and installing the toolkit in editable mode.

Steps:

1. Set `REPO_URL` below to your GitHub repo URL.
2. Run the bootstrap cell once.
3. After that, the rest of the notebook can import `portfolio_toolkit` normally.

If you are running locally, the same cell will automatically fall back to the repository on your machine.


In [ ]:
# Colab / local bootstrap cell
# - In Colab: clone the repo, install the package, and point repo_root at /content/...
# - Locally: just point repo_root at this repository on disk

import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_URL = 'https://github.com/<your-user-or-org>/<your-repo>.git'  # replace with your real repo URL
    REPO_DIR = '/content/Portfolio-Optimizer'

    if '<your-user-or-org>' in REPO_URL or '<your-repo>' in REPO_URL:
        raise ValueError('Set REPO_URL to your real GitHub repository URL before running this cell in Colab.')

    !rm -rf "$REPO_DIR"
    !git clone "$REPO_URL" "$REPO_DIR"
    %cd "$REPO_DIR"
    %pip install -e ".[dev]"

    repo_root = Path(REPO_DIR).resolve()
else:
    repo_root = Path(repo_root).resolve() if 'repo_root' in globals() else Path('../../').resolve()

print('repo_root =', repo_root)


## What This Example Is Doing

This notebook uses `shared_set_2` by default instead of `shared_set_1`.

Why:

- `shared_set_1` is the full S&P 500 universe, so the first download is much larger.
- `shared_set_2` is smaller and faster for a first MLP example.

Once you understand the pattern, you can switch `dataset_name` to `shared_set_1` and run the exact same workflow over a much broader universe.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from portfolio_toolkit import (
    build_features,
    build_metrics,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_portfolio,
    log_predictions,
    make_forward_alpha_target,
    make_forward_realized_vol_target,
    make_forward_return_target,
    slice_split,
    split_dates,
    start_run,
    validate_prediction_frame,
    validate_weights_frame,
    weights_from_predictions_risk_adjusted,
    backtest_weights,
    write_backtest_artifacts,
)

# ---------------------------------------------------------------------
# Basic run configuration.
# ---------------------------------------------------------------------
# repo_root:
#   Where the toolkit config files, cache, MLflow folder, and runs folder live.
# dataset_name:
#   Which shared ticker universe + split rules we want to use.
# horizon:
#   Our prediction horizon in trading days.
# output_dir:
#   Where this notebook will write artifacts like metrics and QuantStats.
# ---------------------------------------------------------------------

repo_root = Path(repo_root).resolve() if 'repo_root' in globals() else Path('../../').resolve()
dataset_name = 'shared_set_2'
model_name = 'torch_mlp_forecast_example'
horizon = 5
output_dir = repo_root / 'runs' / 'mlp_end_to_end_workflow'
output_dir.mkdir(parents=True, exist_ok=True)

spec = get_dataset_spec(dataset_name, repo_root=repo_root)
splits = split_dates(dataset_name, repo_root=repo_root)

print('Dataset preset:', dataset_name)
print('Dataset display name:', spec.name)
print('Number of modeled tickers:', len(spec.tickers))
print('Benchmark ticker:', spec.benchmark_ticker)
print('Train/Val/Test windows:', splits)

## 1. Load Shared Price Data

The toolkit's `load_prices(...)` function is the shared data entrypoint.

What it does:

- reads the selected dataset preset from `configs/datasets.toml`
- downloads daily OHLCV data with `yfinance` if it is not already cached
- always includes `SPY` as the benchmark series
- validates and normalizes the dataframe before returning it

This is one of the main standardization points in the repo: everyone starts from the same dataset preset and the same split boundaries.

In [ ]:
prices = load_prices(dataset_name, repo_root=repo_root)

print('Price frame shape:', prices.shape)
print('Date range:', prices['date'].min(), '->', prices['date'].max())
print('Number of unique tickers in price frame:', prices['ticker'].nunique())
display(prices.head())

## 2. Add Built-In Toolkit Features

We start with a moderate built-in feature set.

These are all created by the shared library, which means other teammates can use the same starting point if they want:

- momentum
- volatility
- RSI
- moving-average distance
- volume z-score
- benchmark-relative return
- intraday range and beta vs SPY

This is a good example of the intended workflow:

- shared features for consistency and speed
- custom notebook-local features on top when you want to experiment

In [ ]:
base_feature_names = [
    'return_5d',
    'return_20d',
    'vol_20d',
    'momentum_20d',
    'momentum_60d',
    'rsi_14',
    'price_to_sma_20d',
    'price_to_sma_50d',
    'volume_zscore_20d',
    'excess_return_20d_vs_spy',
    'intraday_range',
    'beta_20d_spy',
]

base_features = build_features(prices, feature_names=base_feature_names)
print('Base feature frame shape:', base_features.shape)
display(base_features)

## 3. Add New Custom Features In The Notebook

This is where developers keep their freedom.

The toolkit does **not** force everyone to only use built-in features. A normal workflow is:

1. build a shared baseline feature set
2. add experimental features locally in the notebook
3. keep using the shared validation / portfolio / backtest layer afterward

Below we add a few simple handcrafted features:

- `mom_vol_ratio`
  A momentum-to-volatility ratio. This is a quick-and-dirty risk-adjusted trend signal.
- `trend_spread`
  The gap between short-term and medium-term trend distance.
- `quality_signal`
  A simple benchmark-relative momentum signal penalized by volatility.
- `range_volume_interaction`
  A rough interaction term between price range expansion and unusual volume.

In [ ]:
frame = base_features.copy()

# Add a very small constant anywhere we divide so we do not create infinities.
eps = 1e-6

frame['mom_vol_ratio'] = frame['momentum_20d'] / (frame['vol_20d'].abs() + eps)
frame['trend_spread'] = frame['price_to_sma_20d'] - frame['price_to_sma_50d']
frame['quality_signal'] = frame['excess_return_20d_vs_spy'] - 0.5 * frame['vol_20d']
frame['range_volume_interaction'] = frame['intraday_range'] * frame['volume_zscore_20d']

custom_feature_names = [
    'mom_vol_ratio',
    'trend_spread',
    'quality_signal',
    'range_volume_interaction',
]

all_feature_names = base_feature_names + custom_feature_names
display(frame.loc[:, ['date', 'ticker'] + custom_feature_names].head())

## 4. Build Targets

We are going to fit three separate small MLPs with the exact same input features:

- one model for `expected_return`
- one model for `expected_alpha`
- one model for `expected_volatility`

This is not required, but it demonstrates the richer prediction contract supported by the toolkit.

Target builders used here:

- `make_forward_return_target(...)`
- `make_forward_alpha_target(...)`
- `make_forward_realized_vol_target(...)`

In [ ]:
return_targets = make_forward_return_target(prices, horizon=horizon)
alpha_targets = make_forward_alpha_target(prices, horizon=horizon)
vol_targets = make_forward_realized_vol_target(prices, window=horizon)

target_frame = frame.merge(return_targets, on=['date', 'ticker'], how='left')
target_frame = target_frame.merge(alpha_targets, on=['date', 'ticker'], how='left')
target_frame = target_frame.merge(vol_targets, on=['date', 'ticker'], how='left')

# Drop rows only after all features and targets are assembled.
# This is the usual notebook pattern because long-window features and forward targets
# naturally create missing values near the beginning and end of each ticker history.
target_frame = target_frame.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

return_target_col = f'forward_return_{horizon}d'
alpha_target_col = f'forward_alpha_{horizon}d_vs_spy'
vol_target_col = f'forward_realized_vol_{horizon}d'

print('Modeling frame shape after dropping nulls:', target_frame.shape)
display(target_frame.head())

## 5. Shared Train / Validation / Test Splits And Feature Scaling

This section does two very important things:

1. Uses the repo's shared split functions so the notebook respects the official date windows.
2. Standardizes features **using only the training split statistics**.

That second part matters a lot.

We do **not** want to normalize using future information from validation or test rows. So we compute mean and standard deviation from the train split only, then reuse those values everywhere else.

In [ ]:
train = slice_split(target_frame, dataset_name, 'train', repo_root=repo_root)
val = slice_split(target_frame, dataset_name, 'val', repo_root=repo_root)
test = slice_split(target_frame, dataset_name, 'test', repo_root=repo_root)

print('Train rows:', len(train))
print('Val rows:', len(val))
print('Test rows:', len(test))

train_means = train[all_feature_names].mean()
train_stds = train[all_feature_names].std(ddof=0).replace(0.0, 1.0)

def standardize(feature_frame: pd.DataFrame) -> np.ndarray:
    standardized = (feature_frame[all_feature_names] - train_means) / train_stds
    return standardized.to_numpy(dtype=float)

X_train = standardize(train)
X_val = standardize(val)
X_test = standardize(test)

y_train_return = train[return_target_col].to_numpy(dtype=float)
y_val_return = val[return_target_col].to_numpy(dtype=float)
y_test_return = test[return_target_col].to_numpy(dtype=float)

y_train_alpha = train[alpha_target_col].to_numpy(dtype=float)
y_val_alpha = val[alpha_target_col].to_numpy(dtype=float)

y_train_vol = train[vol_target_col].to_numpy(dtype=float)
y_val_vol = val[vol_target_col].to_numpy(dtype=float)

print('X_train shape:', X_train.shape)
print('Feature count:', X_train.shape[1])

## 6. Define A Very Basic MLP Class

This is a tiny PyTorch implementation of a feed-forward neural network for regression.

It is intentionally simple so the workflow is easy to understand:

- fully connected layers
- ReLU hidden activations
- linear output layer
- mean squared error loss
- mini-batch gradient descent
- validation loss tracking

This is **not** meant to be the most advanced PyTorch training setup. It is here so the notebook shows a real neural-network workflow while still staying readable.

In a real team workflow, this cell is exactly the part a researcher would replace with their own model implementation.

In [ ]:
import random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


class TorchMLPRegressor(nn.Module):
    """A very small MLP regressor implemented in PyTorch.

    This keeps the same basic shape as the earlier NumPy example:
    - any number of hidden layers through `hidden_dims`
    - ReLU hidden activations
    - linear output layer for regression
    - mini-batch training with MSE loss

    Example:
    - hidden_dims=(32,) -> one hidden layer
    - hidden_dims=(64, 32) -> two hidden layers
    """

    def __init__(
        self,
        input_dim,
        hidden_dims=(64, 32),
        learning_rate=1e-3,
        epochs=40,
        batch_size=1024,
        random_state=42,
        device=None,
    ):
        super().__init__()

        self.input_dim = int(input_dim)
        self.hidden_dims = tuple(int(x) for x in hidden_dims)
        self.learning_rate = float(learning_rate)
        self.epochs = int(epochs)
        self.batch_size = int(batch_size)
        self.random_state = int(random_state)
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))

        # Keep training reproducible.
        random.seed(self.random_state)
        np.random.seed(self.random_state)
        torch.manual_seed(self.random_state)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.random_state)

        # Build a list like [input_dim, hidden_1, hidden_2, ..., 1]
        layer_dims = [self.input_dim, *self.hidden_dims, 1]
        layers = []

        for layer_idx, (in_dim, out_dim) in enumerate(zip(layer_dims[:-1], layer_dims[1:])):
            linear = nn.Linear(in_dim, out_dim)

            # Xavier initialization keeps early activations in a reasonable scale.
            nn.init.xavier_uniform_(linear.weight)
            nn.init.zeros_(linear.bias)

            layers.append(linear)

            # Final layer stays linear because this is a regression problem.
            if layer_idx < len(layer_dims) - 2:
                layers.append(nn.ReLU())

        self.network = nn.Sequential(*layers).to(self.device)
        self.loss_fn = nn.MSELoss()
        self.optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)

    def forward(self, X):
        return self.network(X)

    def predict(self, X):
        """Return predictions as a 1D NumPy array."""
        self.eval()
        X_tensor = torch.as_tensor(X, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            preds = self.forward(X_tensor).squeeze(-1)
        return preds.cpu().numpy()

    def fit(self, X_train, y_train, X_val=None, y_val=None):
        """Train the model and return a pandas history DataFrame."""
        X_train = torch.as_tensor(X_train, dtype=torch.float32)
        y_train = torch.as_tensor(np.asarray(y_train).reshape(-1, 1), dtype=torch.float32)

        train_dataset = TensorDataset(X_train, y_train)
        train_loader = DataLoader(
            train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
        )

        history = []

        for epoch in range(self.epochs):
            self.train()

            for X_batch, y_batch in train_loader:
                X_batch = X_batch.to(self.device)
                y_batch = y_batch.to(self.device)

                self.optimizer.zero_grad()
                y_pred = self.forward(X_batch)
                loss = self.loss_fn(y_pred, y_batch)
                loss.backward()
                self.optimizer.step()

            train_pred = self.predict(X_train.numpy())
            train_loss = float(np.mean((train_pred - y_train.numpy().reshape(-1)) ** 2))

            row = {"epoch": epoch + 1, "train_loss": train_loss}

            if X_val is not None and y_val is not None:
                y_val_arr = np.asarray(y_val, dtype=float).reshape(-1)
                val_pred = self.predict(X_val)
                val_loss = float(np.mean((val_pred - y_val_arr) ** 2))
                row["val_loss"] = val_loss

            history.append(row)

        return pd.DataFrame(history)



## 7. Train Three Small MLPs

We are using the same feature matrix to train three related regressors:

- return model
- alpha model
- volatility model

This is a very common pattern in research projects:

- one common feature pipeline
- multiple prediction heads or target-specific models

The code below wraps the repeated steps in a helper function so the notebook stays readable.

In [ ]:
def train_single_target_model(target_name, y_train, y_val):
    model = TorchMLPRegressor(
        input_dim=X_train.shape[1],
        hidden_dims=(64, 32),
        learning_rate=1e-3,
        epochs=35,
        batch_size=1024,
        random_state=42,
    )
    history = model.fit(X_train, y_train, X_val=X_val, y_val=y_val)
    best_val_loss = float(history['val_loss'].min()) if 'val_loss' in history else float('nan')
    print(f'{target_name}: best validation loss = {best_val_loss:.8f}')
    return model, history

return_model, return_history = train_single_target_model('expected_return', y_train_return, y_val_return)
alpha_model, alpha_history = train_single_target_model('expected_alpha', y_train_alpha, y_val_alpha)
vol_model, vol_history = train_single_target_model('expected_volatility', y_train_vol, y_val_vol)

training_summary = pd.DataFrame(
    {
        'return_val_loss': return_history['val_loss'],
        'alpha_val_loss': alpha_history['val_loss'],
        'vol_val_loss': vol_history['val_loss'],
    }
)
display(training_summary.tail())

## 8. Create The Standardized Prediction Table

Now we convert raw model outputs into the shared prediction contract used by the rest of the toolkit.

Required columns:

- `date`
- `ticker`
- `horizon`
- `expected_return`

Optional columns we will also populate:

- `expected_alpha`
- `expected_volatility`
- `uncertainty`

For this notebook, `uncertainty` is just a simple constant based on validation RMSE for the return model. That is not a sophisticated uncertainty estimate. It is only here to demonstrate where that information would live in the shared schema.

In [ ]:
test_return_pred = return_model.predict(X_test)
test_alpha_pred = alpha_model.predict(X_test)
test_vol_pred = np.clip(vol_model.predict(X_test), 1e-4, None)

# A toy notebook-level uncertainty estimate: use the square root of the best return-model val MSE.
# In a more serious project this could come from an ensemble, dropout approximation,
# quantile model, Bayesian model, or any custom confidence logic.
return_rmse = float(np.sqrt(return_history['val_loss'].min()))

predictions = test.loc[:, ['date', 'ticker']].copy()
predictions['horizon'] = horizon
predictions['expected_return'] = test_return_pred
predictions['expected_alpha'] = test_alpha_pred
predictions['expected_volatility'] = test_vol_pred
predictions['uncertainty'] = return_rmse

predictions = validate_prediction_frame(
    predictions,
    dataset_name=dataset_name,
    horizon=horizon,
    repo_root=repo_root,
)

display(predictions.head())
print('Prediction frame shape:', predictions.shape)

## 9. Turn Predictions Into A Portfolio Object

The toolkit separates forecasting from portfolio construction.

Here we use the built-in `weights_from_predictions_risk_adjusted(...)` helper.

What it does:

- uses `expected_return / expected_volatility` as the score
- keeps the allocation long-only
- normalizes the scores so each row sums to `1.0`
- returns a `PortfolioWeights` object

This is a good default for demonstrations because it uses more of the prediction contract than a plain top-k rule.

In [ ]:
portfolio = weights_from_predictions_risk_adjusted(
    predictions,
    dataset_name=dataset_name,
    strategy_name=model_name,
)

# The builder already validates internally, but doing it explicitly here makes the notebook
# clearer for new teammates.
validated_weights = validate_weights_frame(
    portfolio.weights,
    dataset_name=dataset_name,
    repo_root=repo_root,
)

print('Strategy name:', portfolio.strategy_name)
print('Weights frame shape:', validated_weights.shape)
display(validated_weights.head())

## 10. Run The Shared Backtest

This is where the toolkit gives you the most value as shared infrastructure.

The backtest layer will:

- load the shared dataset prices
- align rebalance dates to the next available trading day
- apply transaction costs from the dataset preset
- compare the strategy to benchmarks like `SPY`
- compute NAV, returns, turnover, and summary metrics

Because this is shared across the team, different notebooks remain comparable even if the model logic is very different.

In [ ]:
result = backtest_weights(dataset_name, portfolio, repo_root=repo_root)
metrics = build_metrics(result)
artifact_paths = write_backtest_artifacts(result, output_dir)

metrics_table = pd.DataFrame(
    [{'metric': key, 'value': value} for key, value in sorted(metrics.items())]
).sort_values('metric').reset_index(drop=True)

display(metrics_table)
print('QuantStats report:', artifact_paths['quantstats_report'])

## 11. Log The Run To MLflow

The toolkit keeps MLflow intentionally lightweight:

- local SQLite backend
- local artifact storage
- notebook-friendly logging helpers

The pattern here is the one you want teammates to reuse:

1. initialize MLflow locally
2. start a run with meaningful tags
3. log predictions, portfolio weights, and backtest results
4. let MLflow keep the artifact trail

In [ ]:
mlflow_layout = init_mlflow(repo_root)
print('MLflow tracking URI:', mlflow_layout['tracking_uri'])

with start_run(
    run_name=model_name,
    dataset_name=dataset_name,
    tags={
        'workflow': 'mlp_end_to_end_workflow',
        'model_family': 'mlp',
        'prediction_horizon': str(horizon),
    },
    repo_root=repo_root,
):
    import mlflow

    mlflow.log_params(
        {
            'model_name': model_name,
            'dataset_name': dataset_name,
            'horizon': horizon,
            'feature_count': len(all_feature_names),
            'base_feature_list': ','.join(base_feature_names),
            'custom_feature_list': ','.join(custom_feature_names),
            'hidden_dims': '64,32',
            'learning_rate': 1e-3,
            'epochs': 35,
            'batch_size': 1024,
            'portfolio_builder': 'weights_from_predictions_risk_adjusted',
            'cost_bps': spec.cost_bps,
        }
    )

    log_predictions(predictions)
    log_portfolio(portfolio)
    log_backtest(result)

print('MLflow logging complete.')

## 12. Inspect Results

At this point the notebook has produced:

- validated predictions
- validated portfolio weights
- a shared backtest result
- standardized performance metrics
- a QuantStats HTML report
- an MLflow run with artifacts and metrics

That is the full intended research loop for this repo.

In [ ]:
print('Top-level metrics:')
for key, value in sorted(result.metrics.items()):
    print(f'  {key}: {value:.6f}')

print('\nArtifact paths:')
for key, value in artifact_paths.items():
    print(f'  {key}: {value}')

display(result.nav.tail().to_frame('nav'))
display(result.returns.tail().to_frame('returns'))
display(result.turnover.tail().to_frame('turnover'))

In [ ]:
# ---------------------------------------------------------------------
# Final validation cell.
# ---------------------------------------------------------------------
# These checks are intentionally simple. They are the kind of sanity checks
# you want at the end of a notebook before you trust the output.
# ---------------------------------------------------------------------

assert {'total_return', 'annual_return', 'sharpe', 'max_drawdown'}.issubset(result.metrics)
assert validated_weights.index.is_monotonic_increasing
assert (validated_weights.sum(axis=1).round(6) == 1.0).all()
assert Path(artifact_paths['quantstats_report']).exists()
assert {'date', 'ticker', 'horizon', 'expected_return'}.issubset(predictions.columns)

print('End-to-end MLP workflow validated successfully.')